# AI-RISA Ledger Visibility Notebook

This notebook provides a lightweight, read-only dashboard for operational visibility into the AI-RISA prediction and reconciliation ledgers.

**Data sources:**
- `prediction_ledger.jsonl`
- `result_ledger.jsonl`
- `reconciliation_ledger.jsonl`
- `ledger_health_snapshot.json`
- `calibration_patch_v1.json`

**Sections:**
1. Import Required Libraries and Load Data
2. Display Current Health Summary
3. Visualize Reconciliation Trend
4. Show Matchup Family Snapshot Counts
5. Analyze Version Breakdowns

In [ ]:
# 1. Import Required Libraries and Load Data
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

# File paths
prediction_ledger_path = Path('prediction_ledger.jsonl')
result_ledger_path = Path('result_ledger.jsonl')
reconciliation_ledger_path = Path('reconciliation_ledger.jsonl')
ledger_health_snapshot_path = Path('ledger_health_snapshot.json')
calibration_patch_path = Path('calibration_patch_v1.json')

# Load JSONL files into DataFrames
def load_jsonl(path):
    if not path.exists():
        return pd.DataFrame()
    with open(path, 'r', encoding='utf-8') as f:
        lines = [json.loads(line) for line in f if line.strip()]
    return pd.DataFrame(lines)

prediction_ledger = load_jsonl(prediction_ledger_path)
result_ledger = load_jsonl(result_ledger_path)
reconciliation_ledger = load_jsonl(reconciliation_ledger_path)

# Load JSON files
def load_json(path):
    if not path.exists():
        return {}
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

ledger_health_snapshot = load_json(ledger_health_snapshot_path)
calibration_patch = load_json(calibration_patch_path)

# Preview loaded data
print('Prediction Ledger:', prediction_ledger.shape)
print('Result Ledger:', result_ledger.shape)
print('Reconciliation Ledger:', reconciliation_ledger.shape)
print('Ledger Health Snapshot:', bool(ledger_health_snapshot))
print('Calibration Patch:', bool(calibration_patch))

## 2. Display Current Health Summary

This section displays key metrics from `ledger_health_snapshot.json`, including ledger counts, unreconciled predictions, and any health warnings.

In [ ]:
# Display health summary from ledger_health_snapshot
from pprint import pprint

if ledger_health_snapshot:
    print('--- Ledger Health Snapshot ---')
    pprint(ledger_health_snapshot)
else:
    print('No ledger_health_snapshot.json data found.')

## 3. Visualize Reconciliation Trend

This section aggregates and plots reconciliation activity over time using `reconciliation_ledger.jsonl` to show trends in reconciliations.

In [ ]:
# Plot reconciliation activity over time
if not reconciliation_ledger.empty and 'reconciliation_timestamp' in reconciliation_ledger.columns:
    reconciliation_ledger['reconciliation_date'] = pd.to_datetime(reconciliation_ledger['reconciliation_timestamp']).dt.date
    trend = reconciliation_ledger.groupby('reconciliation_date').size()
    plt.figure(figsize=(10,4))
    trend.plot(kind='bar')
    plt.title('Reconciliation Activity Over Time')
    plt.xlabel('Date')
    plt.ylabel('Reconciliations')
    plt.tight_layout()
    plt.show()
else:
    print('No reconciliation data or missing timestamps.')

## 4. Show Matchup Family Snapshot Counts

This section summarizes and displays counts of prediction families and their statuses using `prediction_ledger.jsonl` and `result_ledger.jsonl`.

In [ ]:
# Summarize prediction families and their statuses
if not prediction_ledger.empty and 'prediction_family_id' in prediction_ledger.columns:
    family_counts = prediction_ledger['prediction_family_id'].value_counts()
    print('Prediction Family Counts:')
    display(family_counts)
    
    # Optionally, join with results to show resolved/unresolved
    if not result_ledger.empty and 'prediction_family_id' in result_ledger.columns:
        merged = prediction_ledger.merge(result_ledger[['prediction_family_id']], on='prediction_family_id', how='left', indicator=True)
        merged['status'] = merged['_merge'].map({'both': 'resolved', 'left_only': 'unresolved', 'right_only': 'unknown'})
        status_counts = merged['status'].value_counts()
        print('Prediction Family Status Counts:')
        display(status_counts)
else:
    print('No prediction family data available.')

## 5. Analyze Version Breakdowns

This section breaks down predictions and results by version, showing counts and accuracy metrics over time.

In [ ]:
# Analyze version breakdowns and accuracy metrics
def plot_version_breakdown(df, version_col, label):
    if version_col in df.columns:
        counts = df[version_col].value_counts().sort_index()
        counts.plot(kind='bar', figsize=(8,3), title=f'{label} Version Breakdown')
        plt.xlabel('Version')
        plt.ylabel('Count')
        plt.show()
    else:
        print(f'No {version_col} in {label} ledger.')

# Prediction version breakdown
plot_version_breakdown(prediction_ledger, 'calibration_version', 'Prediction')

# Result version breakdown
plot_version_breakdown(result_ledger, 'calibration_version', 'Result')

# Accuracy over time (if possible)
if not prediction_ledger.empty and not result_ledger.empty and 'prediction_family_id' in prediction_ledger.columns and 'prediction_family_id' in result_ledger.columns:
    merged = prediction_ledger.merge(result_ledger, on='prediction_family_id', suffixes=('_pred', '_res'))
    if 'winner_pred' in merged.columns and 'winner_res' in merged.columns:
        merged['correct'] = merged['winner_pred'] == merged['winner_res']
        acc_by_version = merged.groupby('calibration_version_pred')['correct'].mean()
        acc_by_version.plot(kind='bar', figsize=(8,3), title='Winner Accuracy by Calibration Version')
        plt.xlabel('Calibration Version')
        plt.ylabel('Accuracy')
        plt.show()
    else:
        print('Winner columns not found for accuracy calculation.')
else:
    print('Not enough data for accuracy breakdown.')